# Chapter 25: Real-world Case Studies and Applied Pandas Examples Across Domains**Companion notebook** for *Beginner's Guide to Pandas* by Ravi ShankarRun each cell in order. Exercises are at the end.

In [ ]:
import pandas as pdimport numpy as np

# Real-world Case Studies and Applied Pandas Examples Across DomainsPandas excels at solving practical data problems across diverse industries. This chapter explores how to apply pandas techniques to authentic scenarios you'll encounter in business, science, and engineering contexts. Each case study builds from data creation through analysis to actionable insights, demonstrating patterns you can adapt to your own projects.## E-Commerce: Customer Purchase Analysis### ScenarioAn online retailer wants to understand customer behavior patterns, identify high-value customers, and track revenue trends to optimize inventory and marketing strategies.

In [ ]:
import pandas as pdimport numpy as npfrom datetime import datetime, timedeltaimport matplotlib.pyplot as plt# Create sample e-commerce transaction datanp.random.seed(42)dates = pd.date_range('2023-01-01', periods=1000, freq='D')transactions = pd.DataFrame({    'date': np.random.choice(dates, 500),    'customer_id': np.random.randint(1000, 1500, 500),    'product_category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Sports'], 500),    'quantity': np.random.randint(1, 10, 500),    'unit_price': np.random.uniform(10, 500, 500)})# Calculate total transaction valuetransactions['total_value'] = transactions['quantity'] * transactions['unit_price']# Sort by date for time-series analysistransactions = transactions.sort_values('date').reset_index(drop=True)print(transactions.head())

### Customer SegmentationIdentify customer tiers based on purchase behavior:

In [ ]:
# Aggregate customer metricscustomer_metrics = transactions.groupby('customer_id').agg({    'total_value': ['sum', 'mean', 'count'],    'date': ['min', 'max']}).round(2)# Flatten column namescustomer_metrics.columns = ['total_spent', 'avg_order_value', 'purchase_count',                            'first_purchase', 'last_purchase']# Calculate how long each customer has been activecustomer_metrics['days_active'] = (    customer_metrics['last_purchase'] - customer_metrics['first_purchase']).dt.days# Segment customers into tiers based on total spendingcustomer_metrics['tier'] = pd.qcut(    customer_metrics['total_spent'],    q=3,    labels=['Bronze', 'Silver', 'Gold'])print(customer_metrics.head(10))print("\nCustomer Tier Distribution:")print(customer_metrics['tier'].value_counts())

### Monthly Revenue TrendsAnalyze revenue patterns by month and category:

In [ ]:
# Extract year-month period from datetransactions['year_month'] = transactions['date'].dt.to_period('M')# Monthly revenue by categorymonthly_revenue = transactions.groupby(    ['year_month', 'product_category'])['total_value'].sum().unstack(fill_value=0)print("Monthly Revenue by Category:")print(monthly_revenue)# Visualize trendsmonthly_revenue.plot(kind='line', figsize=(12, 6), marker='o')plt.title('Monthly Revenue by Product Category')plt.xlabel('Month')plt.ylabel('Revenue ($)')plt.legend(title='Category')plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()# Calculate month-over-month growthtotal_monthly = transactions.groupby('year_month')['total_value'].sum()mom_growth = total_monthly.pct_change() * 100print("\nMonth-over-Month Growth (%):")print(mom_growth)

---## Healthcare: Patient Readmission Analysis### ScenarioA hospital system needs to identify patients at high risk of readmission within 30 days to implement preventive interventions.

In [ ]:
# Create patient admission recordspatient_data = pd.DataFrame({    'patient_id': range(1000, 1100),    'admission_date': pd.date_range('2023-01-01', periods=100),    'discharge_date': pd.date_range('2023-01-01', periods=100) + timedelta(days=5),    'age': np.random.randint(18, 90, 100),    'primary_diagnosis': np.random.choice(        ['Pneumonia', 'Heart Failure', 'COPD', 'Diabetes'], 100    ),    'comorbidity_count': np.random.randint(0, 6, 100),    'readmitted': np.random.choice([True, False], 100, p=[0.25, 0.75])})# Calculate length of staypatient_data['length_of_stay'] = (    patient_data['discharge_date'] - patient_data['admission_date']).dt.daysprint(patient_data.head())

### Risk Factor AnalysisIdentify which factors correlate with readmission:

In [ ]:
# Analyze readmission rates by diagnosisreadmission_by_diagnosis = patient_data.groupby('primary_diagnosis').agg({    'readmitted': ['sum', 'count', 'mean']}).round(3)readmission_by_diagnosis.columns = ['readmissions', 'total_patients', 'readmission_rate']readmission_by_diagnosis = readmission_by_diagnosis.sort_values(    'readmission_rate', ascending=False)print("Readmission Rates by Diagnosis:")print(readmission_by_diagnosis)# Age-based risk stratificationpatient_data['age_group'] = pd.cut(    patient_data['age'],    bins=[0, 30, 50, 70, 100],    labels=['18-30', '31-50', '51-70', '70+'])age_readmission = patient_data.groupby('age_group')['readmitted'].agg(    ['sum', 'count', 'mean'])age_readmission.columns = ['readmissions', 'total_patients', 'readmission_rate']print("\nReadmission Rates by Age Group:")print(age_readmission)

### Risk Score and VisualizationBuild a composite risk score and visualize the key drivers:

In [ ]:
# Visualize risk factorsfig, axes = plt.subplots(1, 2, figsize=(14, 5))readmission_by_diagnosis['readmission_rate'].plot(    kind='barh', ax=axes[0], color='coral')axes[0].set_title('Readmission Rate by Diagnosis')axes[0].set_xlabel('Readmission Rate')age_readmission['readmission_rate'].plot(    kind='bar', ax=axes[1], color='skyblue')axes[1].set_title('Readmission Rate by Age Group')axes[1].set_ylabel('Readmission Rate')axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)plt.tight_layout()plt.show()# Create a composite risk score based on clinical factorspatient_data['risk_score'] = (    (patient_data['age'] > 65).astype(int) * 2 +    (patient_data['comorbidity_count'] > 3).astype(int) * 3 +    (patient_data['length_of_stay'] > 5).astype(int) * 2)high_risk = patient_data[    patient_data['risk_score'] >= 4].sort_values('risk_score', ascending=False)print(f"\nHigh-Risk Patients: {len(high_risk)} out of {len(patient_data)}")print(high_risk[['patient_id', 'age', 'primary_diagnosis',                 'comorbidity_count', 'risk_score']].head(10))

---## Finance: Stock Portfolio Performance Tracking### ScenarioAn investment firm tracks multiple stock holdings and needs to monitor portfolio composition, individual returns, and overall performance.

In [ ]:
# Create portfolio snapshot dataportfolio = pd.DataFrame({    'ticker': ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA'],    'shares_owned': [100, 50, 25, 15, 40],    'purchase_price': [150.25, 280.50, 2800.00, 3200.00, 850.00],    'current_price': [175.50, 310.25, 3100.00, 3450.00, 920.00]})# Calculate portfolio metricsportfolio['purchase_value'] = portfolio['shares_owned'] * portfolio['purchase_price']portfolio['current_value'] = portfolio['shares_owned'] * portfolio['current_price']portfolio['gain_loss'] = portfolio['current_value'] - portfolio['purchase_value']portfolio['return_pct'] = (    portfolio['gain_loss'] / portfolio['purchase_value'] * 100).round(2)# Portfolio allocationportfolio['allocation_pct'] = (    portfolio['current_value'] / portfolio['current_value'].sum() * 100).round(2)print("Portfolio Analysis:")print(portfolio[['ticker', 'shares_owned', 'current_price',                 'current_value', 'return_pct', 'allocation_pct']])print(f"\nTotal Portfolio Value: ${portfolio['current_value'].sum():,.2f}")print(f"Total Gain/Loss: ${portfolio['gain_loss'].sum():,.2f}")print(    f"Overall Return: "    f"{(portfolio['gain_loss'].sum() / portfolio['purchase_value'].sum() * 100):.2f}%")# Visualize allocationportfolio.set_index('ticker')['allocation_pct'].plot(    kind='pie', autopct='%1.1f%%', figsize=(8, 6))plt.title('Portfolio Allocation by Ticker')plt.ylabel('')plt.show()

### Daily Returns and CorrelationsSimulate price history to analyze return statistics and inter-stock correlations:

In [ ]:
# Simulate daily price history for the full yearprice_dates = pd.date_range('2023-01-01', '2023-12-31', freq='D')portfolio_history = pd.DataFrame({    'AAPL':  np.random.normal(150, 5, len(price_dates)).cumsum() + 150,    'GOOGL': np.random.normal(100, 4, len(price_dates)).cumsum() + 100,    'MSFT':  np.random.normal(300, 8, len(price_dates)).cumsum() + 300,    'TSLA':  np.random.normal(200, 15, len(price_dates)).cumsum() + 200}, index=price_dates)# Calculate daily returnsdaily_returns = portfolio_history.pct_change()# Summary statisticsportfolio_stats = pd.DataFrame({    'Mean Daily Return':    daily_returns.mean(),    'Volatility':           daily_returns.std(),    'Annualized Return':    daily_returns.mean() * 252,    'Annualized Volatility': daily_returns.std() * np.sqrt(252)})print("Portfolio Statistics:")print(portfolio_stats.round(4))# Correlation matrixprint("\nCorrelation Matrix:")print(daily_returns.corr().round(3))

---## Marketing: Campaign Performance Analysis### ScenarioA marketing team runs campaigns across multiple channels and needs to evaluate effectiveness by channel and audience demographic.

In [ ]:
# Campaign performance datanp.random.seed(42)campaign_data = pd.DataFrame({    'campaign_id':     range(1, 101),    'channel':         np.random.choice(['Email', 'Social', 'Display', 'Search'], 100),    'target_age_group': np.random.choice(['18-25', '26-35', '36-50', '50+'], 100),    'budget':          np.random.uniform(1000, 50000, 100),    'impressions':     np.random.randint(10000, 500000, 100),    'clicks':          np.random.randint(100, 10000, 100),    'conversions':     np.random.randint(5, 1000, 100),    'revenue':         np.random.uniform(500, 50000, 100)})# Calculate key performance metricscampaign_data['ctr'] = (    campaign_data['clicks'] / campaign_data['impressions']).round(4)campaign_data['conversion_rate'] = (    campaign_data['conversions'] / campaign_data['clicks']).round(4)campaign_data['roi'] = (    (campaign_data['revenue'] - campaign_data['budget']) / campaign_data['budget']).round(3)campaign_data['cpc'] = (    campaign_data['budget'] / campaign_data['clicks']).round(2)

### Channel PerformanceAggregate metrics at the channel level to compare overall effectiveness:

In [ ]:
# Performance by channelchannel_performance = campaign_data.groupby('channel').agg({    'budget':      'sum',    'impressions': 'sum',    'clicks':      'sum',    'conversions': 'sum',    'revenue':     'sum'})channel_performance['avg_ctr'] = (    channel_performance['clicks'] / channel_performance['impressions']).round(4)channel_performance['channel_roi'] = (    (channel_performance['revenue'] - channel_performance['budget']) /    channel_performance['budget']).round(3)print("Channel Performance Summary:")print(channel_performance)

### Demographic InsightsAnalyze campaign effectiveness across audience segments:

In [ ]:
# ROI by demographicroi_by_demographic = campaign_data.groupby('target_age_group').agg({    'revenue':     'sum',    'budget':      'sum',    'conversions': 'sum'})roi_by_demographic['roi'] = (    (roi_by_demographic['revenue'] - roi_by_demographic['budget']) /    roi_by_demographic['budget']).round(3)print("ROI by Age Group:")print(roi_by_demographic)# Multi-dimensional view: revenue by age group and channeldemographic_analysis = campaign_data.pivot_table(    values=['revenue', 'conversions', 'budget'],    index='target_age_group',    columns='channel',    aggfunc='sum')print("\nRevenue by Age Group and Channel:")print(demographic_analysis['revenue'].round(0))

---## Environmental Science: Air Quality Monitoring### ScenarioAn environmental agency monitors air quality across multiple stations and needs to classify pollution levels, identify problem periods, and detect seasonal trends.

In [ ]:
# Air quality measurements from multiple stationsnp.random.seed(42)dates = pd.date_range('2023-01-01', periods=365, freq='D')air_quality = pd.DataFrame({    'date':    np.tile(dates, 3),    'station': np.repeat(['Downtown', 'Suburban', 'Rural'], 365),    'pm25':    np.random.normal(35, 15, 1095),    'pm10':    np.random.normal(50, 20, 1095),    'no2':     np.random.normal(45, 10, 1095),    'o3':      np.random.normal(60, 15, 1095)})# Ensure non-negative valuesair_quality[['pm25', 'pm10', 'no2', 'o3']] = (    air_quality[['pm25', 'pm10', 'no2', 'o3']].clip(lower=0))print(air_quality.head(10))

### Air Quality ClassificationCategorize pollution levels using EPA PM2.5 thresholds and identify problem periods:

In [ ]:
# Define AQI categories based on PM2.5def classify_air_quality(pm25):    if pm25 <= 12:        return 'Good'    elif pm25 <= 35.4:        return 'Moderate'    elif pm25 <= 55.4:        return 'Unhealthy for Sensitive Groups'    elif pm25 <= 150.4:        return 'Unhealthy'    else:        return 'Very Unhealthy'air_quality['aqi_category'] = air_quality['pm25'].apply(classify_air_quality)# Monthly statistics by stationmonthly_stats = air_quality.groupby(    [pd.Grouper(key='date', freq='ME'), 'station']).agg({    'pm25': ['mean', 'max', 'min'],    'pm10': 'mean',    'no2':  'mean',    'o3':   'mean'}).round(2)print("Monthly Air Quality Statistics:")print(monthly_stats)# Count unhealthy days by stationunhealthy_days = (    air_quality[        air_quality['aqi_category'].isin(['Unhealthy', 'Very Unhealthy'])    ]    .groupby('station')    .size())print("\nDays with Unhealthy Air Quality by Station:")print(unhealthy_days)

### Trend AnalysisVisualize weekly-averaged pollutant levels across stations:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, pollutant in enumerate(['pm25', 'pm10', 'no2', 'o3']):
    ax = axes[idx // 2, idx % 2]
    for station in air_quality['station'].unique():
        station_data = air_quality[air_quality['station'] == station]
        weekly_avg = station_data.set_index('date')[pollutant].resample('W').mean()
        ax.plot(weekly_avg.index, weekly_avg.values,
                label=station, marker='o', alpha=0.7)
    ax.set_title(f'{pollutant.upper()} Levels Over Time')
    ax.set_ylabel('Concentration Level')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Monthly trend for PM2.5
air_quality['year_month'] = air_quality['date'].dt.to_period('M')
monthly_pm25 = air_quality.groupby('year_month')['pm25'].mean().to_frame()
monthly_pm25['pm25_change'] = monthly_pm25['pm25'].pct_change().round(3)

print("\nMonthly PM2.5 Trend:")
print(monthly_pm25)

---## Educational Institution Analytics### ScenarioA university wants to identify at-risk students early and understand which factors are most associated with retention.

In [ ]:
# Create student performance datasetnp.random.seed(42)students = pd.DataFrame({    'student_id':        range(1000, 1200),    'enrollment_year':   np.random.choice([2021, 2022, 2023], 200),    'major':             np.random.choice(['Engineering', 'Business', 'Arts', 'Sciences'], 200),    'gpa':               np.random.normal(3.2, 0.5, 200).clip(0, 4.0),    'attendance_rate':   np.random.normal(0.85, 0.15, 200).clip(0, 1),    'library_visits':    np.random.randint(0, 100, 200),    'tutoring_sessions': np.random.randint(0, 20, 200),    'retained':          np.random.choice([True, False], 200, p=[0.75, 0.25])})print(students.head())

### Retention AnalysisIdentify factors associated with student retention:

In [ ]:
# Performance by majormajor_stats = students.groupby('major').agg({    'gpa':            'mean',    'attendance_rate': 'mean',    'retained':       'mean',    'student_id':     'count'}).round(3)major_stats.columns = ['avg_gpa', 'avg_attendance', 'retention_rate', 'student_count']major_stats = major_stats.sort_values('retention_rate', ascending=False)print("Performance Metrics by Major:")print(major_stats)# Identify at-risk students using multiple criteriaat_risk = students[    (students['gpa'] < 2.5) |    (students['attendance_rate'] < 0.75) |    (students['tutoring_sessions'] == 0)]print(f"\nAt-Risk Students Identified: {len(at_risk)} ({len(at_risk)/len(students)*100:.1f}%)")print("\nAt-Risk Student Breakdown by Major:")print(at_risk.groupby('major').size())# Compare retained vs. non-retained studentsretention_comparison = students.groupby('retained')[    ['gpa', 'attendance_rate', 'library_visits', 'tutoring_sessions']].mean()retention_comparison.index = ['Not Retained', 'Retained']print("\nComparison: Retained vs. Not Retained Students")print(retention_comparison.round(2))

### Visualizing Retention Drivers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))# GPA distribution by retentionstudents.boxplot(column='gpa', by='retained', ax=axes[0, 0])axes[0, 0].set_title('GPA Distribution by Retention Status')axes[0, 0].set_xlabel('Retained')# Attendance by retentionstudents.boxplot(column='attendance_rate', by='retained', ax=axes[0, 1])axes[0, 1].set_title('Attendance Rate by Retention Status')axes[0, 1].set_xlabel('Retained')# Scatter: GPA vs Attendance coloured by retentionfor retained_status in [True, False]:    subset = students[students['retained'] == retained_status]    axes[1, 0].scatter(        subset['gpa'], subset['attendance_rate'],        label='Retained' if retained_status else 'Not Retained',        alpha=0.6    )axes[1, 0].set_xlabel('GPA')axes[1, 0].set_ylabel('Attendance Rate')axes[1, 0].set_title('GPA vs. Attendance Rate')axes[1, 0].legend()axes[1, 0].grid(True, alpha=0.3)# Tutoring sessions by retentionstudents.boxplot(column='tutoring_sessions', by='retained', ax=axes[1, 1])axes[1, 1].set_title('Tutoring Sessions by Retention Status')axes[1, 1].set_xlabel('Retained')plt.suptitle('')plt.tight_layout()plt.show()

---## Data Quality and Preprocessing PatternsReal datasets are rarely clean. This section demonstrates practical strategies for handling missing values, a challenge that appears in every domain covered above.### Handling Missing Data

In [ ]:
# Create a dataset with realistic missing valuesnp.random.seed(42)messy_data = pd.DataFrame({    'transaction_id': range(1, 101),    'amount': [        np.nan if np.random.random() < 0.1 else np.random.uniform(10, 1000)        for _ in range(100)    ],    'category': np.random.choice(['A', 'B', 'C', None], 100),    'date': pd.date_range('2023-01-01', periods=100)})print("Missing Values Summary:")print(messy_data.isnull().sum())# Strategy 1: Forward fill — appropriate for time-ordered datafilled_forward = messy_data.copy()filled_forward['amount'] = filled_forward['amount'].ffill()# Strategy 2: Fill with the group mean — preserves category-level patternsfilled_mean = messy_data.copy()filled_mean['amount'] = filled_mean.groupby('category')['amount'].transform(    lambda x: x.fillna(x.mean()))# Strategy 3: Drop incomplete records — use when missingness is lowclean_data = messy_data.dropna()print(f"\nOriginal records:       {len(messy_data)}")print(f"After removing nulls:   {len(clean_data)}")print(f"Data retention:         {len(clean_data)/len(messy_data)*100:.1f}%")

Choosing the right strategy depends on context. Forward fill works well for sensor readings or stock prices where the previous value is a reasonable estimate. Group-mean imputation is better when values vary systematically by category. Dropping rows is safest when the missing fraction is small and the remaining data is still representative.---## Key TakeawaysThese case studies demonstrate pandas' versatility across domains:- **Data aggregation** simplifies complex analyses through `groupby()` and `agg()`- **Time-series functionality** enables trend detection and temporal analysis- **Flexible calculations** allow creation of custom metrics and risk scores- **Filtering and comparison** identify outliers and anomalies efficiently- **Pivot tables** reveal patterns across multiple dimensions- **Domain expertise matters**: understanding your data's context enables better analysis choices- **Combine multiple techniques**: real problems rarely require just one pandas operation- **Handle missing data thoughtfully**: different strategies suit different scenariosBy adapting these patterns to your specific domain, you can leverage pandas to extract meaningful insights from your data.

---# ExercisesTest your understanding of this chapter's concepts.

### Exercise 1: E-Commerce Customer SegmentationAnalyze a small e-commerce dataset to compute per-customer purchase statistics. Calculate total spend, average order value, and order count for each customer, then classify them into 'High', 'Medium', or 'Low' value segments based on total spend.

In [ ]:
import pandas as pd# Sample e-commerce orders dataorders = pd.DataFrame({    'order_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],    'customer_id': ['C01', 'C02', 'C01', 'C03', 'C02', 'C03', 'C01', 'C04', 'C04', 'C02'],    'order_value': [120.50, 340.00, 89.99, 450.00, 210.75, 130.00, 300.00, 95.00, 410.00, 175.50]})# TODO: Group by customer_id and compute:#   - total_spend (sum of order_value)#   - avg_order_value (mean of order_value)#   - order_count (count of order_id)customer_stats = None# TODO: Reset the index so customer_id becomes a regular column# TODO: Add a 'segment' column using pd.cut() or np.select() / pd.cut with bins:#   High  -> total_spend >= 500#   Medium -> 200 <= total_spend < 500#   Low   -> total_spend < 200customer_stats['segment'] = Noneprint(customer_stats)

### Exercise 2: Patient Readmission Risk FlaggingWork with a simulated patient dataset to identify high-risk readmission candidates. Compute each patient's average length of stay and number of visits, then flag patients who have been admitted more than twice or whose average stay exceeds 5 days as 'High Risk'.

In [ ]:
import pandas as pd# Sample patient admissions dataadmissions = pd.DataFrame({    'patient_id': ['P001', 'P002', 'P001', 'P003', 'P002', 'P001', 'P004', 'P003', 'P004', 'P002'],    'admission_date': pd.to_datetime([        '2024-01-05', '2024-01-10', '2024-02-15', '2024-01-20',        '2024-02-28', '2024-03-10', '2024-01-18', '2024-03-05',        '2024-02-22', '2024-03-30'    ]),    'length_of_stay': [3, 7, 6, 2, 5, 8, 4, 3, 9, 6]  # days})# TODO: Group by patient_id and compute:#   - visit_count: number of admissions#   - avg_stay: mean length_of_staypatient_summary = None# TODO: Reset the index# TODO: Create a boolean column 'high_risk' that is True when#       visit_count > 2 OR avg_stay > 5patient_summary['high_risk'] = None# TODO: Print only the high-risk patientsprint(patient_summary)

### Exercise 3: Stock Portfolio Performance TrackerGiven a small stock portfolio dataset with daily closing prices, calculate each stock's daily return, cumulative return over the period, and identify which stock delivered the best overall performance. Use percentage change and cumulative product operations.

In [ ]:
import pandas as pd# Daily closing prices for a small portfolioprices = pd.DataFrame({    'date': pd.date_range(start='2024-01-01', periods=6, freq='B'),    'AAPL':  [185.0, 187.5, 186.0, 190.0, 192.5, 195.0],    'MSFT':  [375.0, 378.0, 374.0, 380.0, 383.0, 379.0],    'GOOGL': [140.0, 142.0, 141.5, 145.0, 144.0, 148.0]})prices = prices.set_index('date')# TODO: Calculate daily percentage returns for each stock using pct_change()daily_returns = None# TODO: Calculate the cumulative return for each stock.#       Hint: cumulative return = (1 + daily_return).cumprod() - 1#       Drop NaN rows before computing.cumulative_returns = None# TODO: Find the final cumulative return (last row) for each stock#       and identify the best performing stock (use idxmax())final_returns = Nonebest_stock = Noneprint("Daily Returns:")print(daily_returns.round(4))print("\nFinal Cumulative Returns:")print(final_returns.round(4))print(f"\nBest performing stock: {best_stock}")

### Exercise 4: Air Quality Data Cleaning and SummaryProcess a messy air quality monitoring dataset that contains missing values and duplicate records. Clean the data by removing duplicates and filling missing AQI readings with the daily mean, then produce a summary showing the average, max, and min AQI per city.

In [ ]:
import pandas as pd# Raw air quality readings (contains duplicates and missing values)raw_data = pd.DataFrame({    'city':       ['Austin', 'Austin', 'Austin', 'Denver', 'Denver', 'Denver', 'Austin', 'Denver', 'Seattle', 'Seattle', 'Seattle'],    'date':       ['2024-03-01', '2024-03-01', '2024-03-02', '2024-03-01', '2024-03-02',                   '2024-03-03', '2024-03-03', '2024-03-03', '2024-03-01', '2024-03-02', '2024-03-03'],    'aqi':        [45, 45, None, 60, 75, None, 55, 80, 35, None, 50]})# TODO: Remove duplicate rows (keep the first occurrence)clean_data = None# TODO: Fill missing 'aqi' values with the mean aqi of the same city.#       Hint: use groupby + transform('mean') to compute group means,#       then use fillna()clean_data['aqi'] = None# TODO: Convert the 'date' column to datetimeclean_data['date'] = None# TODO: Group by 'city' and compute avg_aqi, max_aqi, min_aqicity_summary = Noneprint("Cleaned Data:")print(clean_data)print("\nCity AQI Summary:")print(city_summary)

---# Solutions*Scroll down only after you've attempted the exercises above.*<br><br><br><br><br><br><br><br><br><br>

### Solution 1: E-Commerce Customer Segmentation

In [ ]:
import pandas as pdimport numpy as np# Sample e-commerce orders dataorders = pd.DataFrame({    'order_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],    'customer_id': ['C01', 'C02', 'C01', 'C03', 'C02', 'C03', 'C01', 'C04', 'C04', 'C02'],    'order_value': [120.50, 340.00, 89.99, 450.00, 210.75, 130.00, 300.00, 95.00, 410.00, 175.50]})# Group by customer_id and compute aggregated statisticscustomer_stats = orders.groupby('customer_id').agg(    total_spend=('order_value', 'sum'),    avg_order_value=('order_value', 'mean'),    order_count=('order_id', 'count')).reset_index()# Add a segment column based on total_spendconditions = [    customer_stats['total_spend'] >= 500,    (customer_stats['total_spend'] >= 200) & (customer_stats['total_spend'] < 500),    customer_stats['total_spend'] < 200]choices = ['High', 'Medium', 'Low']customer_stats['segment'] = np.select(conditions, choices)print(customer_stats)

### Solution 2: Patient Readmission Risk Flagging

In [ ]:
import pandas as pdimport numpy as np# Sample patient admissions dataadmissions = pd.DataFrame({    'patient_id': ['P001', 'P002', 'P001', 'P003', 'P002', 'P001', 'P004', 'P003', 'P004', 'P002'],    'admission_date': pd.to_datetime([        '2024-01-05', '2024-01-10', '2024-02-15', '2024-01-20',        '2024-02-28', '2024-03-10', '2024-01-18', '2024-03-05',        '2024-02-22', '2024-03-30'    ]),    'length_of_stay': [3, 7, 6, 2, 5, 8, 4, 3, 9, 6]  # days})# Group by patient_id and compute visit count and average staypatient_summary = admissions.groupby('patient_id').agg(    visit_count=('admission_date', 'count'),    avg_stay=('length_of_stay', 'mean')).reset_index()# Flag high-risk patientspatient_summary['high_risk'] = (    (patient_summary['visit_count'] > 2) | (patient_summary['avg_stay'] > 5))# Print only high-risk patientsprint(patient_summary[patient_summary['high_risk'] == True])

### Solution 3: Stock Portfolio Performance Tracker

In [ ]:
import pandas as pdimport numpy as np# Daily closing prices for a small portfolioprices = pd.DataFrame({    'date': pd.date_range(start='2024-01-01', periods=6, freq='B'),    'AAPL':  [185.0, 187.5, 186.0, 190.0, 192.5, 195.0],    'MSFT':  [375.0, 378.0, 374.0, 380.0, 383.0, 379.0],    'GOOGL': [140.0, 142.0, 141.5, 145.0, 144.0, 148.0]})prices = prices.set_index('date')# Calculate daily percentage returnsdaily_returns = prices.pct_change()# Calculate cumulative returns, dropping the first NaN rowcumulative_returns = (1 + daily_returns.dropna()).cumprod() - 1# Get the final cumulative return for each stockfinal_returns = cumulative_returns.iloc[-1]# Identify the best performing stockbest_stock = final_returns.idxmax()print("Daily Returns:")print(daily_returns.round(4))print("\nFinal Cumulative Returns:")print(final_returns.round(4))print(f"\nBest performing stock: {best_stock}")

### Solution 4: Air Quality Data Cleaning and Summary

In [ ]:
import pandas as pdimport numpy as np# Raw air quality readings (contains duplicates and missing values)raw_data = pd.DataFrame({    'city':       ['Austin', 'Austin', 'Austin', 'Denver', 'Denver', 'Denver', 'Austin', 'Denver', 'Seattle', 'Seattle', 'Seattle'],    'date':       ['2024-03-01', '2024-03-01', '2024-03-02', '2024-03-01', '2024-03-02',                   '2024-03-03', '2024-03-03', '2024-03-03', '2024-03-01', '2024-03-02', '2024-03-03'],    'aqi':        [45, 45, None, 60, 75, None, 55, 80, 35, None, 50]})# Remove duplicate rowsclean_data = raw_data.drop_duplicates().copy()# Fill missing aqi values with the mean aqi of the same citycity_mean_aqi = clean_data.groupby('city')['aqi'].transform('mean')clean_data['aqi'] = clean_data['aqi'].fillna(city_mean_aqi)# Convert date column to datetimeclean_data['date'] = pd.to_datetime(clean_data['date'])# Group by city and compute summary statisticscity_summary = clean_data.groupby('city').agg(    avg_aqi=('aqi', 'mean'),    max_aqi=('aqi', 'max'),    min_aqi=('aqi', 'min')).reset_index()print("Cleaned Data:")print(clean_data)print("\nCity AQI Summary:")print(city_summary)